# VedOCR: Custom OCR Model Training for Sanskrit Manuscripts

**Author:** Suyash Kumar Bhagat  
**Institution:** Delhi Technological University

This notebook trains a custom OCR model on MorphBG-binarised Sanskrit manuscript data.

## Architecture
- **Encoder:** ResNet-based CNN for visual feature extraction
- **Decoder:** BiLSTM + CTC for sequence prediction
- **Loss:** CTC (Connectionist Temporal Classification)

## Setup
Run this notebook on Google Colab with GPU enabled for faster training.

In [ ]:
# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

# Check GPU availability
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q opencv-python-headless pillow numpy
!pip install -q python-Levenshtein editdistance
!pip install -q tqdm matplotlib seaborn
!pip install -q tensorboard

print("Dependencies installed")

## 2. Mount Google Drive (if on Colab)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Update these paths to point to your data on Google Drive
    DATA_ROOT = '/content/drive/MyDrive/vedocr_project'
    print(f"Data root: {DATA_ROOT}")
else:
    DATA_ROOT = '.'
    print("Using local directory")

## 3. Load and Prepare Dataset

In [ ]:
import json
import cv2
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from typing import List, Tuple, Dict
import random

# Load parsed annotations
annotations_file = Path(DATA_ROOT) / 'parsed_annotations' / 'parsed_annotations.json'

with open(annotations_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

lines = data['lines']
character_inventory = data['character_inventory']

print(f"Loaded {len(lines)} annotated lines")
print(f"Character inventory: {len(character_inventory)} unique glyphs")

# Show sample
sample = lines[0]
print(f"\nSample line:")
print(f"  Text: {sample['text']}")
print(f"  Image: {sample['line_image_path']}")

## 4. Create Character Vocabulary

In [ ]:
class CharacterVocabulary:
    """Character-to-index mapping for CTC."""
    
    def __init__(self, characters: List[str]):
        # CTC blank token is index 0
        self.blank_idx = 0
        
        # Map characters to indices (starting from 1)
        self.char_to_idx = {char: idx + 1 for idx, char in enumerate(characters)}
        self.idx_to_char = {idx: char for char, idx in self.char_to_idx.items()}
        self.idx_to_char[0] = '[BLANK]'
        
        self.vocab_size = len(characters) + 1  # +1 for blank
        
    def encode(self, text: str) -> List[int]:
        """Convert text to indices."""
        return [self.char_to_idx.get(char, self.blank_idx) for char in text]
    
    def decode(self, indices: List[int]) -> str:
        """Convert indices to text."""
        # Remove blank tokens and consecutive duplicates (CTC decode)
        chars = []
        prev_idx = None
        
        for idx in indices:
            if idx != self.blank_idx and idx != prev_idx:
                chars.append(self.idx_to_char.get(idx, ''))
            prev_idx = idx
            
        return ''.join(chars)

# Create vocabulary
vocab = CharacterVocabulary(character_inventory)
print(f"Vocabulary size: {vocab.vocab_size} (including CTC blank)")

# Test encoding/decoding
test_text = sample['text'][:20]
encoded = vocab.encode(test_text)
decoded = vocab.decode(encoded)
print(f"\nTest encode/decode:")
print(f"  Original: {test_text}")
print(f"  Encoded:  {encoded[:10]}...")
print(f"  Decoded:  {decoded}")

## 5. Create PyTorch Dataset

In [ ]:
class SanskritLineDataset(Dataset):
    """Dataset for Sanskrit manuscript line images."""
    
    def __init__(
        self,
        lines: List[Dict],
        vocab: CharacterVocabulary,
        img_height: int = 64,
        img_width: int = 1024,
        augment: bool = False
    ):
        self.lines = lines
        self.vocab = vocab
        self.img_height = img_height
        self.img_width = img_width
        self.augment = augment
        
    def __len__(self):
        return len(self.lines)
    
    def __getitem__(self, idx):
        line_data = self.lines[idx]
        
        # Load image
        img_path = line_data['line_image_path']
        if IN_COLAB:
            # Adjust path for Colab
            img_path = str(Path(DATA_ROOT) / Path(img_path).name)
        
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if image is None:
            # Fallback: create blank image
            image = np.ones((self.img_height, self.img_width), dtype=np.uint8) * 255
        
        # Resize to fixed height, maintain aspect ratio up to max width
        h, w = image.shape
        scale = self.img_height / h
        new_w = min(int(w * scale), self.img_width)
        
        image = cv2.resize(image, (new_w, self.img_height))
        
        # Pad to fixed width
        if new_w < self.img_width:
            pad_w = self.img_width - new_w
            image = np.pad(image, ((0, 0), (0, pad_w)), constant_values=255)
        
        # Normalize to [0, 1] and invert (0=background, 1=foreground)
        image = image.astype(np.float32) / 255.0
        image = 1.0 - image  # Invert: dark ink becomes high value
        
        # Add channel dimension
        image = image[np.newaxis, :, :]  # (1, H, W)
        
        # Encode label
        text = line_data['text']
        label = self.vocab.encode(text)
        
        return {
            'image': torch.FloatTensor(image),
            'label': torch.LongTensor(label),
            'text': text,
            'label_length': len(label)
        }

# Split into train/val
random.seed(42)
random.shuffle(lines)

split_idx = int(0.9 * len(lines))
train_lines = lines[:split_idx]
val_lines = lines[split_idx:]

print(f"Train set: {len(train_lines)} lines")
print(f"Val set:   {len(val_lines)} lines")

# Create datasets
train_dataset = SanskritLineDataset(train_lines, vocab, augment=True)
val_dataset = SanskritLineDataset(val_lines, vocab, augment=False)

# Test dataset
sample_item = train_dataset[0]
print(f"\nSample batch item:")
print(f"  Image shape: {sample_item['image'].shape}")
print(f"  Label length: {sample_item['label_length']}")
print(f"  Text: {sample_item['text']}")

## 6. Define OCR Model Architecture

In [ ]:
class CRNN(nn.Module):
    """Convolutional Recurrent Neural Network for OCR."""
    
    def __init__(
        self,
        img_height: int,
        num_classes: int,
        cnn_output_height: int = 1,
        rnn_hidden: int = 256,
        num_rnn_layers: int = 2
    ):
        super(CRNN, self).__init__()
        
        self.img_height = img_height
        self.num_classes = num_classes
        
        # CNN backbone (similar to ResNet)
        self.cnn = nn.Sequential(
            # Conv block 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # H/2
            
            # Conv block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # H/4
            
            # Conv block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),  # H/8, W unchanged
            
            # Conv block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),  # H/16, W unchanged
            
            # Final conv to reduce height further
            nn.Conv2d(512, 512, kernel_size=2, stride=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        
        # Calculate CNN output feature size
        self.cnn_output_height = cnn_output_height
        self.rnn_input_size = 512 * cnn_output_height
        
        # Bidirectional LSTM
        self.rnn = nn.LSTM(
            input_size=self.rnn_input_size,
            hidden_size=rnn_hidden,
            num_layers=num_rnn_layers,
            bidirectional=True,
            batch_first=False
        )
        
        # Linear layer for classification
        self.fc = nn.Linear(rnn_hidden * 2, num_classes)  # *2 for bidirectional
        
    def forward(self, x):
        # x: (batch, 1, H, W)
        
        # CNN
        conv_out = self.cnn(x)  # (batch, 512, H', W')
        
        # Reshape for RNN: (W', batch, features)
        b, c, h, w = conv_out.size()
        conv_out = conv_out.permute(3, 0, 1, 2)  # (W', batch, 512, H')
        conv_out = conv_out.reshape(w, b, c * h)  # (W', batch, 512*H')
        
        # RNN
        rnn_out, _ = self.rnn(conv_out)  # (W', batch, rnn_hidden*2)
        
        # Classification
        output = self.fc(rnn_out)  # (W', batch, num_classes)
        output = F.log_softmax(output, dim=2)
        
        return output

# Initialize model
model = CRNN(
    img_height=64,
    num_classes=vocab.vocab_size,
    cnn_output_height=3,
    rnn_hidden=256,
    num_rnn_layers=2
).to(device)

print(f"Model initialized on {device}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Training Setup

In [ ]:
import Levenshtein
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

def collate_fn(batch):
    """Collate function for DataLoader (handles variable-length sequences)."""
    images = torch.stack([item['image'] for item in batch], dim=0)
    labels = [item['label'] for item in batch]
    label_lengths = torch.LongTensor([item['label_length'] for item in batch])
    texts = [item['text'] for item in batch]
    
    # Pad labels
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=0)
    
    return images, labels_padded, label_lengths, texts

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

# CTC Loss
criterion = nn.CTCLoss(blank=vocab.blank_idx, zero_infinity=True)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    verbose=True
)

print("Training setup complete")

## 8. Training Functions

In [ ]:
def decode_predictions(outputs, vocab):
    """Decode CTC outputs to text."""
    # outputs: (T, batch, num_classes)
    _, preds = outputs.max(2)  # (T, batch)
    preds = preds.transpose(1, 0).contiguous()  # (batch, T)
    
    pred_texts = []
    for pred in preds:
        pred_text = vocab.decode(pred.cpu().numpy())
        pred_texts.append(pred_text)
    
    return pred_texts

def compute_cer(pred, gt):
    """Compute Character Error Rate."""
    if len(gt) == 0:
        return 1.0 if len(pred) > 0 else 0.0
    return Levenshtein.distance(pred, gt) / len(gt)

def train_epoch(model, train_loader, criterion, optimizer, vocab, device):
    """Train for one epoch."""
    model.train()
    
    total_loss = 0
    cer_scores = []
    
    pbar = tqdm(train_loader, desc="Training")
    
    for images, labels, label_lengths, texts in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward
        outputs = model(images)  # (T, batch, num_classes)
        
        # Compute input lengths (after CNN downsampling)
        T = outputs.size(0)
        input_lengths = torch.full((images.size(0),), T, dtype=torch.long)
        
        # CTC loss
        loss = criterion(outputs, labels, input_lengths, label_lengths)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        # Metrics
        total_loss += loss.item()
        
        # Decode predictions for CER
        pred_texts = decode_predictions(outputs, vocab)
        for pred, gt in zip(pred_texts, texts):
            cer_scores.append(compute_cer(pred, gt))
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'cer': f'{np.mean(cer_scores):.4f}'
        })
    
    avg_loss = total_loss / len(train_loader)
    avg_cer = np.mean(cer_scores)
    
    return avg_loss, avg_cer

def validate(model, val_loader, criterion, vocab, device):
    """Validate model."""
    model.eval()
    
    total_loss = 0
    cer_scores = []
    
    with torch.no_grad():
        for images, labels, label_lengths, texts in tqdm(val_loader, desc="Validation"):
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward
            outputs = model(images)
            
            # Compute input lengths
            T = outputs.size(0)
            input_lengths = torch.full((images.size(0),), T, dtype=torch.long)
            
            # Loss
            loss = criterion(outputs, labels, input_lengths, label_lengths)
            total_loss += loss.item()
            
            # CER
            pred_texts = decode_predictions(outputs, vocab)
            for pred, gt in zip(pred_texts, texts):
                cer_scores.append(compute_cer(pred, gt))
    
    avg_loss = total_loss / len(val_loader)
    avg_cer = np.mean(cer_scores)
    
    return avg_loss, avg_cer

print("Training functions defined")

## 9. Train Model

In [ ]:
# Training configuration
NUM_EPOCHS = 50
BEST_CER = 1.0

# Training history
history = {
    'train_loss': [],
    'train_cer': [],
    'val_loss': [],
    'val_cer': []
}

# Training loop
for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"{'='*60}")
    
    # Train
    train_loss, train_cer = train_epoch(
        model, train_loader, criterion, optimizer, vocab, device
    )
    
    # Validate
    val_loss, val_cer = validate(model, val_loader, criterion, vocab, device)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_cer'].append(train_cer)
    history['val_loss'].append(val_loss)
    history['val_cer'].append(val_cer)
    
    # Print results
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f}  |  Train CER: {train_cer:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}  |  Val CER:   {val_cer:.4f}")
    
    # Save best model
    if val_cer < BEST_CER:
        BEST_CER = val_cer
        checkpoint_path = Path(DATA_ROOT) / 'models' / 'best_model.pth'
        checkpoint_path.parent.mkdir(exist_ok=True, parents=True)
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_cer': val_cer,
            'vocab': vocab
        }, checkpoint_path)
        
        print(f"  *** Best model saved (CER: {val_cer:.4f}) ***")

print("\nTraining complete!")

## 10. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CTC Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# CER
axes[1].plot(history['train_cer'], label='Train CER')
axes[1].plot(history['val_cer'], label='Val CER')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Character Error Rate')
axes[1].set_title('Training and Validation CER')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(Path(DATA_ROOT) / 'results' / 'training_history.png', dpi=150)
plt.show()

print(f"Best validation CER: {BEST_CER:.4f}")

## 11. Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(Path(DATA_ROOT) / 'models' / 'best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model (CER: {checkpoint['val_cer']:.4f})")

# Evaluate on validation set
val_loss, val_cer = validate(model, val_loader, criterion, vocab, device)

print(f"\nFinal Validation Results:")
print(f"  Loss: {val_loss:.4f}")
print(f"  CER: {val_cer:.4f}")
print(f"  Accuracy: {(1 - val_cer) * 100:.2f}%")

## 12. Visualize Predictions

In [ ]:
# Get a batch of validation samples
val_iter = iter(val_loader)
images, labels, label_lengths, texts = next(val_iter)

# Predict
model.eval()
with torch.no_grad():
    outputs = model(images.to(device))
    pred_texts = decode_predictions(outputs, vocab)

# Visualize
fig, axes = plt.subplots(4, 2, figsize=(15, 12))
axes = axes.flatten()

for idx in range(min(8, len(images))):
    img = images[idx].squeeze().cpu().numpy()
    gt = texts[idx]
    pred = pred_texts[idx]
    cer = compute_cer(pred, gt)
    
    axes[idx].imshow(img, cmap='gray')
    axes[idx].axis('off')
    axes[idx].set_title(
        f"GT:   {gt[:50]}\n"
        f"Pred: {pred[:50]}\n"
        f"CER: {cer:.3f}",
        fontsize=8,
        loc='left'
    )

plt.tight_layout()
plt.savefig(Path(DATA_ROOT) / 'results' / 'predictions_visualization.png', dpi=150)
plt.show()

## 13. Export Model for Inference

In [ ]:
# Export final model with all necessary components
export_path = Path(DATA_ROOT) / 'models' / 'vedocr_final.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab,
    'model_config': {
        'img_height': 64,
        'img_width': 1024,
        'num_classes': vocab.vocab_size,
        'cnn_output_height': 3,
        'rnn_hidden': 256,
        'num_rnn_layers': 2
    },
    'performance': {
        'best_val_cer': BEST_CER,
        'best_val_accuracy': (1 - BEST_CER) * 100
    }
}, export_path)

print(f"Model exported to: {export_path}")
print(f"\nFinal Performance:")
print(f"  CER: {BEST_CER:.4f}")
print(f"  Accuracy: {(1 - BEST_CER) * 100:.2f}%")

## 14. Save Training Summary

In [ ]:
# Save training summary as JSON
summary = {
    'timestamp': datetime.now().isoformat(),
    'dataset': {
        'num_train_lines': len(train_lines),
        'num_val_lines': len(val_lines),
        'vocab_size': vocab.vocab_size,
        'num_unique_chars': len(character_inventory)
    },
    'model': {
        'architecture': 'CRNN',
        'num_parameters': sum(p.numel() for p in model.parameters()),
        'img_height': 64,
        'img_width': 1024,
        'rnn_hidden': 256,
        'num_rnn_layers': 2
    },
    'training': {
        'num_epochs': NUM_EPOCHS,
        'batch_size': 16,
        'optimizer': 'Adam',
        'learning_rate': 0.001,
        'loss_function': 'CTC'
    },
    'performance': {
        'best_val_cer': float(BEST_CER),
        'best_val_accuracy': float((1 - BEST_CER) * 100),
        'final_train_loss': float(history['train_loss'][-1]),
        'final_val_loss': float(history['val_loss'][-1])
    },
    'history': history
}

summary_path = Path(DATA_ROOT) / 'results' / 'training_summary.json'
summary_path.parent.mkdir(exist_ok=True, parents=True)

with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Training summary saved to: {summary_path}")